# Task 5 — Step 1: In-Domain GSM8K Math Evaluation (RLVR vs. RLAIF)

### Research Question
Holding the policy family and group-based RL procedure approximately fixed, how does the **source of feedback** alter mathematical reasoning behavior?
- **Reinforcement Learning with Verifiable Rewards (RLVR)**: Uses an exact final-answer rule checker: $r_{\text{RLVR}} = \mathbf{1}[\text{verifier}(y) = \text{gold}(x)]$.
- **Reinforcement Learning from AI Feedback (RLAIF)**: Uses a pairwise AI preference judge to assign relative win rates: $r_{\text{RLAIF}}(y_k) = \frac{\text{wins}(y_k) + 0.5\text{ties}(y_k)}{K - 1}$.

### Required Metrics
- Exact final-answer mathematical accuracy.
- Format compliance rate (matching designated answer format).
- Pairwise win rate against SFT under the AI judge.
- Solution response length (reasoning chain verbosity).


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run GSM8K Evaluation or Load Results


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset gsm
else:
    print("[OK] Loading existing in-domain GSM8K results...")


## 2. In-Domain Evaluation Summary Table


In [ ]:
gsm_file = "results/task5_feedback/math_eval_gsm.json"
if os.path.exists(gsm_file):
    with open(gsm_file, encoding='utf-8') as f:
        gsm_data = json.load(f)
        
    pols = gsm_data.get("per_policy", {})
    pairs = gsm_data.get("pairwise_comparisons", {})
    
    rows = []
    for p in ["sft", "rlvr", "rlaif"]:
        st = pols.get(p, {})
        win_info = pairs.get(f"{p}_vs_sft", {})
        win_rate_str = f"{win_info.get('win_rate', 0)*100:.1f}%" if p != "sft" else "Baseline"
        rows.append({
            "Policy": p.upper(),
            "Exact Accuracy (%)": f"{st.get('exact_accuracy', 0)*100:.1f}%",
            "Format Compliance (%)": f"{st.get('format_compliance_rate', 0)*100:.1f}%",
            "Win Rate vs SFT": win_rate_str,
            "Mean Tokens": f"{st.get('mean_tokens', 0):.1f}",
            "Mean Words": f"{st.get('mean_words', 0):.1f}"
        })
    df_gsm = pd.DataFrame(rows)
    display(df_gsm)


## 3. Visualizing In-Domain Performance


In [ ]:
if os.path.exists(gsm_file):
    pol_names = [r["Policy"] for r in rows]
    accs = [float(r["Exact Accuracy (%)"].replace('%','')) for r in rows]
    lens = [float(r["Mean Tokens"]) for r in rows]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    axes[0].bar(pol_names, accs, color=['#64748B', '#10B981', '#3B82F6'], width=0.4, alpha=0.85)
    axes[0].set_ylabel('Exact Answer Accuracy (%)')
    axes[0].set_title('GSM8K Exact Accuracy: RLVR vs. RLAIF vs. SFT')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Length
    axes[1].bar(pol_names, lens, color=['#64748B', '#10B981', '#3B82F6'], width=0.4, alpha=0.85)
    for i, l in enumerate(lens):
        axes[1].text(i, l + 2, f"{l:.1f}", ha='center', fontweight='bold')
    axes[1].set_ylabel('Mean Tokens')
    axes[1].set_title('Reasoning Length Comparison')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()


## 4. Publication Figure Inspection


In [ ]:
p = "report/figures/task5_math_eval_gsm.png"
if os.path.exists(p):
    print(f"Displaying publication figure: {p}")
    display(Image(filename=p, width=650))
